<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-08-agents-and-adk/lesson-8.4-a2a/notebooks/GCP_Capstone_8.4_A2A.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 8.4 A2A — A Peer That Knows DocuMind Only Through the MCP Server: Card, Task, Client, Deploy
**Netsetos GenAI Engineering — GCP Capstone** · Module 8 · rebuilt on the live lane, 8 September 2026

The agent another company would build. It knows DocuMind by one URL, the MCP server 7.2 deployed; speaks to it as its own account; and is itself reachable by other agents over A2A: an agent card, JSON-RPC, tasks. Three protocols meet in one request. This notebook owns the kit's `services/agent` (the peer's code, requirements, Dockerfile) and `commands/lesson-8.4.sh` (the deploy), runs the peer here first, wraps it as an ADK sub-agent, and then reaches the deployed one with a credential minted per request.


## Setup
The kit is cloned because this lesson *writes* a service into it. Nothing is imported from it: the peer is outside the kit by construction.


In [ ]:
!pip install -q "google-adk[a2a,mcp]==2.8.0" "a2a-sdk[http-server]==1.1.2" google-genai==2.22.0 google-auth==2.57.1

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
KIT        = "/content/agentic-ai-weekend-gcp-learners"   # the kit: this lesson WRITES a service into it, and imports nothing from it
BRANCH     = "rag-production-hardening"

import json, os, subprocess, sys, time, uuid
import google.auth
from google.auth import impersonated_credentials
from google.auth.transport.requests import AuthorizedSession, Request

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)

creds, _ = google.auth.default()
NUMBER    = AuthorizedSession(creds).get(f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
MCP_URL   = f"https://documind-mcp-{NUMBER}.{REGION}.run.app"      # 7.2's server: the ONLY thing the peer knows about DocuMind
AGENT_URL = f"https://documind-agent-{NUMBER}.{REGION}.run.app"    # where the peer will live (Cell 7); deterministic like every service's
MEMBER_SA = f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com" # what this notebook mints as: on the rosters, and allowed to invoke the peer

os.environ.update({"GOOGLE_CLOUD_PROJECT": PROJECT_ID, "GOOGLE_CLOUD_LOCATION": "global", "GOOGLE_GENAI_USE_VERTEXAI": "TRUE"})
SCOPE = ["https://www.googleapis.com/auth/cloud-platform"]


def id_token_as(service_account: str, audience: str) -> str:
    """A Google ID token minted AS a service account, for one audience, with the email inside (4.8, 7.3)."""
    source, _ = google.auth.default()
    target = impersonated_credentials.Credentials(source_credentials=source, target_principal=service_account, target_scopes=SCOPE)
    idc = impersonated_credentials.IDTokenCredentials(target, target_audience=audience, include_email=True)
    idc.refresh(Request())
    return idc.token


print("MCP:", MCP_URL, "\npeer (once deployed):", AGENT_URL)


## Cell 1: The peer's code
An ADK agent whose only tools are the MCP server's, with the credential minted per call, exposed with `to_a2a`. Written into the kit at the path it deploys from.


In [ ]:
AGENT_PY = r'''
"""DocuMind's A2A peer - an agent OUTSIDE the kit, reaching the lane only through documind-mcp.

Lesson 8.4. Module 7 ended on one sentence: the MCP server is for agents outside the kit, and
the brains inside the kit call retrieve() directly. This service is that sentence with an
address. It is the agent another team - another company - would build: it knows DocuMind by
ONE URL (MCP_URL), speaks to it as its own account (documind-agent-sa, on one roster), and is
itself reachable by other agents over A2A: an agent card at /.well-known/agent-card.json and
JSON-RPC at /. Three protocols meet in one request:

    A2A client --ID token for THIS service--> here --McpToolset, ID token for documind-mcp--> documind-mcp
                                                                 (verifies agent-sa, checks the roster)
                                                                     --retrieve() as documind-mcp-sa--> documind-api

What is deliberately NOT here: `from shared import ...`. No retrieve(), no roster, no verifier.
The image copies only this directory (Dockerfile) and the gate in tools/check_auth_wiring.py
fails the build if a kit import appears. That absence is the lesson: a peer that imported the
kit would be a fifth brain, not a peer.

Identity, precisely. Cloud Run IAM decides who may call THIS service (roles/run.invoker on
documind-agent: ui-sa and chat-sa). The A2A protocol carries no identity of its own, so the
peer speaks to the lane as ITSELF - never as the caller - and the roster sees documind-agent-sa.
That is why the account sits on exactly one roster (acme, deploy/Makefile `roster`): a peer
scoped to one tenant needs no tenant argument, and a request naming another tenant is refused
by the server, not by this code.

The credential refreshes: `header_provider` is called by ADK on EVERY tool call
(google/adk/tools/mcp_tool/mcp_tool.py, _run_async_impl), so the token is minted fresh each
time - a Cloud Run ID token lasts an hour, and a token captured once is 7.3's bug with an
hour-long fuse. On Cloud Run the metadata server mints it as the service account; in a
notebook, DOCUMIND_IMPERSONATE_SA mints as a roster member (the same hook 7.1 taught).
"""
from __future__ import annotations

import logging
import os
from urllib.parse import urlparse

from google.adk.a2a.utils.agent_to_a2a import to_a2a
from google.adk.agents import LlmAgent
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams
from starlette.requests import Request
from starlette.responses import JSONResponse

logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger("documind.agent")

MCP_URL = os.environ.get("MCP_URL", "").rstrip("/")     # https://documind-mcp-NUMBER.us-central1.run.app
SELF_URL = os.environ.get("SELF_URL", "").rstrip("/")   # this service's URL - the address on its agent card
MODEL = os.environ.get("AGENT_MODEL", "gemini-3.6-flash")
# id-token (Cloud Run, or a notebook with DOCUMIND_IMPERSONATE_SA) | none (the 7.1 local server, no IAM)
MCP_AUTH = os.environ.get("MCP_AUTH", "id-token")

INSTRUCTION = (
    "You are DocuMind, an agent that answers questions about a company's documents. "
    "You hold no documents yourself: for any question about documents call retrieve, answer only "
    "from what it returns, and cite the sources it names. Use list_documents to see what the corpus "
    "holds, corpus_stats for counts, and calculate_processing_cost to price a set of pages. "
    "Do not pass a tenant argument unless the person names a tenant: your own account decides which "
    "corpus is yours, and the server refuses a tenant you are not on. "
    "If a tool refuses, or says the corpus cannot answer, say so plainly and cite nothing."
)


def _mcp_headers(readonly_context=None) -> dict:
    """The credential for documind-mcp, minted per call.

    ADK calls this on every tool call, so nothing here is cached. The audience is the server's
    ROOT url: Cloud Run checks the token was minted for it, and the server checks it again as
    its SELF_URL (7.1). With DOCUMIND_IMPERSONATE_SA set - a notebook, a laptop - the token is
    minted AS that account, which is how a person runs this peer before it has an account.
    """
    if MCP_AUTH == "none":
        return {}
    if not MCP_URL:
        raise RuntimeError("MCP_URL is not configured - the peer has no lane to reach")
    import google.auth
    import google.auth.transport.requests
    import google.oauth2.id_token

    request = google.auth.transport.requests.Request()
    impersonate = os.environ.get("DOCUMIND_IMPERSONATE_SA")
    if impersonate:
        from google.auth import impersonated_credentials
        source, _ = google.auth.default()
        target = impersonated_credentials.Credentials(
            source_credentials=source, target_principal=impersonate,
            target_scopes=["https://www.googleapis.com/auth/cloud-platform"])
        idc = impersonated_credentials.IDTokenCredentials(target, target_audience=MCP_URL, include_email=True)
        idc.refresh(request)
        return {"Authorization": f"Bearer {idc.token}"}
    return {"Authorization": f"Bearer {google.oauth2.id_token.fetch_id_token(request, MCP_URL)}"}


def build_agent() -> LlmAgent:
    """The peer. Its only tools are the lane's, discovered from the server at first use."""
    lane = McpToolset(
        connection_params=StreamableHTTPConnectionParams(
            url=f"{MCP_URL}/mcp",
            # ADK's default HTTP timeout is 5 s. A retrieve() through the server is rag-api plus a
            # Gemini answer - seconds when warm, up to 90 s when the API is cold (7.2) - while a
            # roster refusal is instant. The first live smoke passed the zeta task and failed the
            # acme one on exactly this line. 120 s covers the server's own RAG_TIMEOUT_S=90.
            timeout=120,
            sse_read_timeout=300,
        ),
        header_provider=_mcp_headers,
    )
    return LlmAgent(
        name="documind_peer",
        model=MODEL,
        description="Answers questions about DocuMind's documents with citations, "
                    "through the DocuMind MCP server. Prices document processing.",
        instruction=INSTRUCTION,
        tools=[lane],
    )


def _card_address() -> tuple[str, int, str]:
    """host, port, protocol for the agent card - the deployed URL, not localhost.

    to_a2a builds the card's url as protocol://host:port/ and A2A clients post to exactly that,
    so a card that says localhost is a peer nobody can reach. SELF_URL is the deterministic
    run.app address (deploy/commands/lesson-8.4.sh); unset, the peer is on a laptop.
    """
    if SELF_URL:
        u = urlparse(SELF_URL)
        return u.hostname or "localhost", u.port or (443 if u.scheme == "https" else 80), u.scheme or "https"
    return "localhost", int(os.environ.get("PORT", "8080")), "http"


async def health(request: Request) -> JSONResponse:
    return JSONResponse({"status": "ok", "mcp_url": MCP_URL or None, "self_url": SELF_URL or None,
                         "model": MODEL, "kit_import": False})


agent = build_agent()
_host, _port, _protocol = _card_address()
app = to_a2a(agent, host=_host, port=_port, protocol=_protocol)
app.add_route("/health", health, methods=["GET"])
log.info('{"event":"agent_up","card":"%s://%s:%s/.well-known/agent-card.json","mcp_url":"%s"}',
         _protocol, _host, _port, MCP_URL)

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=int(os.environ.get("PORT", "8080")))
'''

# WRITTEN INTO THE KIT, at the path the kit deploys from. Not imported from it: the peer imports
# nothing under deploy/shared, and the Dockerfile below copies only its own directory. On a fresh
# clone of the branch this write is a no-op, which the diff line proves.
os.makedirs(f"{KIT}/deploy/services/agent", exist_ok=True)
os.chdir(f"{KIT}/deploy/services/agent")
with open('agent.py', 'w') as f:
    f.write(AGENT_PY)
diff = subprocess.run(["git", "-C", KIT, "diff", "--stat", "--", "deploy/services/agent/agent.py"],
                      capture_output=True, text=True).stdout.strip()
print(f"agent.py: {len(AGENT_PY.splitlines())} lines |", "identical to the kit's committed file" if not diff else diff)
assert "from shared" not in AGENT_PY and "import shared" not in AGENT_PY, "the peer must not import the kit"


## Cell 2: Requirements and a Dockerfile that copies nothing from `shared/`


In [ ]:
REQUIREMENTS_TXT = r'''
# DocuMind A2A peer (lesson 8.4): ADK + the A2A server + the MCP client, and nothing from the kit.
# [a2a] brings a2a-sdk[http-server] (the Starlette app to_a2a returns); [mcp] brings the mcp 1.x
# protocol library McpToolset speaks. No fastmcp here - this is a client of 7.2's server, not a server.
google-adk[a2a,mcp]==2.8.0
a2a-sdk[http-server]==1.1.2
google-genai==2.22.0
google-auth==2.57.1
uvicorn==0.52.4
'''

DOCKERFILE = r'''
FROM python:3.12-slim
RUN useradd --create-home --shell /bin/bash --uid 10001 app
WORKDIR /app
# BUILD CONTEXT IS deploy/, like every kit image (cloudbuild.yaml names the Dockerfile) - and
# this one copies ONLY its own directory. No shared/: the peer reaches the lane through the MCP
# server, never through the kit's retrieve(), and the absence is provable from the image:
#     docker run --rm IMAGE sh -c 'ls shared 2>/dev/null || echo "no kit inside"'
COPY services/agent/requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY --chown=app:app services/agent/ ./
USER app
ENV PYTHONUNBUFFERED=1
# Cloud Run sets PORT. The A2A app is stateless per request; tasks live in memory for the
# life of a message, so no --session-affinity is needed.
CMD ["sh", "-c", "uvicorn agent:app --host 0.0.0.0 --port ${PORT:-8080}"]
'''

with open('requirements.txt', 'w') as f:
    f.write(REQUIREMENTS_TXT)
with open('Dockerfile', 'w') as f:
    f.write(DOCKERFILE)
os.chdir("/content")
# The proof that there is no kit inside, from the build recipe itself: one COPY, of this directory.
print("COPY lines:", [l for l in DOCKERFILE.splitlines() if l.startswith("COPY")])
assert not any("shared" in l for l in DOCKERFILE.splitlines() if l.startswith("COPY"))


## Cell 3: Run the peer here
Minting as the notebook's roster member; the MCP server is the deployed one.


In [ ]:
# THE PEER, RUN HERE. Its only configuration is the MCP url and how to mint for it: in this notebook,
# by impersonating a roster member (the kit's own hook, honoured by agent.py); on Cloud Run, from the
# service's own account. No SELF_URL, so the card says localhost - a peer only this notebook can reach.
PEER_PORT = 8001
env = dict(os.environ, MCP_URL=MCP_URL, DOCUMIND_IMPERSONATE_SA=MEMBER_SA, PORT=str(PEER_PORT), PYTHONUNBUFFERED="1")
peer_proc = subprocess.Popen([sys.executable, "-m", "uvicorn", "agent:app", "--host", "127.0.0.1", "--port", str(PEER_PORT), "--log-level", "warning"],
                             cwd=f"{KIT}/deploy/services/agent", env=env)
import httpx
LOCAL = f"http://127.0.0.1:{PEER_PORT}"
for _ in range(60):
    try:
        health = httpx.get(f"{LOCAL}/health", timeout=2).json(); break
    except Exception:
        time.sleep(0.5)
else:
    raise SystemExit("the peer did not come up - read its log above")
print("health:", health)
assert health["kit_import"] is False


## Cell 4: The agent card


In [ ]:
# THE AGENT CARD: how another agent discovers this one. Built by to_a2a from the agent's name,
# description and tools - which is why the skills below are the MCP server's four, learned at
# start-up. A2A 1.x puts the address under supportedInterfaces; a caller posts JSON-RPC to it.
card = httpx.get(f"{LOCAL}/.well-known/agent-card.json", timeout=10).json()
print("name       :", card.get("name"))
print("description:", (card.get("description") or "")[:120])
print("address    :", (card.get("supportedInterfaces") or [{}])[0].get("url"))
print("skills     :", [s.get("id") for s in card.get("skills", [])])
print("streaming  :", (card.get("capabilities") or {}).get("streaming"))


## Cell 5: One task, three protocols
A2A from here to the peer, MCP from the peer to the server, HTTP from the server to the API. Asserted.


In [ ]:
def send(base: str, text: str, client: "httpx.Client | None" = None) -> tuple:
    """message/send over JSON-RPC - the bytes deploy/smoke/smoke_agent.py sends. Returns (task, answer text)."""
    body = {"jsonrpc": "2.0", "id": str(uuid.uuid4()), "method": "message/send",
            "params": {"message": {"role": "user", "kind": "message", "messageId": str(uuid.uuid4()),
                                   "parts": [{"kind": "text", "text": text}]}}}
    r = (client or httpx).post(f"{base}/", json=body, timeout=180)
    j = r.json()
    if "error" in j:
        return j, ""
    task = j["result"]
    texts = [pt.get("text", "") for a in task.get("artifacts", []) for pt in a.get("parts", []) if pt.get("kind") == "text"]
    return task, "\n".join(t for t in texts if t)


# ONE TASK, THREE PROTOCOLS. This cell speaks A2A to the peer; the peer speaks MCP to documind-mcp
# as the notebook's impersonated identity; the server speaks HTTP to documind-api as its own.
# Golden row lk-16: the Payment of Gratuity Act, five years.
task, answer = send(LOCAL, "After how many years of continuous service does gratuity become payable?")
print("task state:", task.get("status", {}).get("state"), "| task id:", task.get("id"))
print("answer    :", answer[:400])
assert "five" in answer.lower() or "5 years" in answer.lower(), "the peer did not answer from the corpus: " + answer[:200]


## Cell 6: An ADK client - the peer as a sub-agent


In [ ]:
from google.adk.agents import LlmAgent
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# AN ADK CLIENT. RemoteA2aAgent wraps the peer as a SUB-AGENT: the root reads its card, and the model
# delegates with transfer_to_agent exactly as it would to a local specialist (8.2). No auth on this
# hop yet - the local peer is unauthenticated; Cell 8 adds the credential for the deployed one.
peer = RemoteA2aAgent(name="documind_peer", description="Answers questions about DocuMind's documents with citations.",
                      agent_card=f"{LOCAL}/.well-known/agent-card.json")
root = LlmAgent(name="orchestrator", model="gemini-3.6-flash",
                instruction="Delegate every question about documents, policies or contracts to documind_peer and relay its answer verbatim.",
                sub_agents=[peer])

session_service = InMemorySessionService()


async def run_turn(target, question: str) -> list:
    runner = Runner(agent=target, app_name="a2a_demo", session_service=session_service)
    session = await session_service.create_session(app_name="a2a_demo", user_id="student")
    trace = []
    async for event in runner.run_async(user_id="student", session_id=session.id,
                                        new_message=types.Content(role="user", parts=[types.Part.from_text(text=question)])):
        for part in (event.content.parts if event.content and event.content.parts else []):
            if part.function_call:
                trace.append(f"[{event.author}] call {part.function_call.name}({dict(part.function_call.args or {})})")
            if part.text and part.text.strip():
                trace.append(f"[{event.author}] {part.text.strip()[:300]}")
    return trace


for line in await run_turn(root, "What is the notice period for a confirmed E3?"):     # lk-06: 60 days
    print(" ", line)


## Cell 7: Deploy - `commands/lesson-8.4.sh`
Run in Cloud Shell from `deploy/`. The DEPLOY block is what `make build deploy-services` runs for this service.


In [ ]:
DEPLOY = r'''
# 1. The image, from the deploy/ context; the Dockerfile copies ONLY services/agent - no shared/.
gcloud builds submit --config=cloudbuild.yaml \
  --substitutions=_IMAGE=${REGION:-us-central1}-docker.pkg.dev/$PROJECT/documind/agent:$GIT_SHA,_DOCKERFILE=services/agent/Dockerfile .

# 2. The service: private (IAM), its own account, the MCP server's url, and its own url for the card.
#    No RAG_API_URL and no roster: the peer does not know the API exists. GOOGLE_CLOUD_LOCATION=global
#    is for Gemini 3.x; SELF_URL is what makes the agent card say a reachable address.
gcloud run deploy documind-agent \
  --image=${REGION:-us-central1}-docker.pkg.dev/$PROJECT/documind/agent:$GIT_SHA \
  --region=${REGION:-us-central1} --platform=managed \
  --no-allow-unauthenticated --ingress=all \
  --memory=1Gi --cpu=1 --concurrency=20 --timeout=300 \
  --min-instances=${MIN_INSTANCES:-0} --max-instances=5 --cpu-boost \
  --execution-environment=gen2 \
  --service-account=documind-agent-sa@$PROJECT.iam.gserviceaccount.com \
  --set-env-vars="^|^GOOGLE_CLOUD_PROJECT=$PROJECT|GOOGLE_GENAI_USE_VERTEXAI=1|GOOGLE_CLOUD_LOCATION=global|MCP_URL=https://documind-mcp-$PROJECT_NUMBER.${REGION:-us-central1}.run.app|SELF_URL=https://documind-agent-$PROJECT_NUMBER.${REGION:-us-central1}.run.app|AGENT_MODEL=gemini-3.6-flash"

# 3. Who may call the peer: the UI's account (this notebook and make smoke-agent mint as it) and the
#    chat account. A2A carries no identity of its own - IAM is the peer's front door, and the peer
#    speaks to the lane as itself whoever called it (sa.tf: documind-agent-sa, on acme's roster only).
for sa in documind-ui-sa documind-chat-sa; do
  gcloud run services add-iam-policy-binding documind-agent \
    --region=${REGION:-us-central1} --project=$PROJECT \
    --member="serviceAccount:$sa@$PROJECT.iam.gserviceaccount.com" --role=roles/run.invoker --quiet
done
'''

SMOKE = r'''
# From deploy/ on a machine with gcloud (Cloud Shell): the card refused without a token, the card
# with one, a task answered through documind-mcp, a task naming zeta refused by the roster and relayed.
make smoke-agent PROJECT=$PROJECT

# By hand: the card, as a caller IAM admits
NUMBER=$(gcloud projects describe $PROJECT --format='value(projectNumber)')
AGENT=https://documind-agent-$NUMBER.${REGION:-us-central1}.run.app
TOKEN=$(gcloud auth print-identity-token --include-email \
  --impersonate-service-account=documind-ui-sa@$PROJECT.iam.gserviceaccount.com --audiences=$AGENT)
curl -s -H "Authorization: Bearer $TOKEN" $AGENT/.well-known/agent-card.json | head -c 400; echo
curl -s -o /dev/null -w "without a token: %{http_code}\n" $AGENT/.well-known/agent-card.json
'''

# These two blocks are the kit's commands/lesson-8.4.sh (deploy/extract_documind.py lifts them from
# this notebook), and `make build deploy-services SERVICES=agent SCRIPTS=commands/lesson-8.4.sh`
# runs the DEPLOY one with PROJECT, GIT_SHA and PROJECT_NUMBER set. Run them in Cloud Shell, from deploy/.
print(DEPLOY)


## Cell 8: The deployed peer, with a credential per request
`httpx.Auth` mints for the peer's URL on every request; `RemoteA2aAgent` uses that client for the card and the tasks.


In [ ]:
DEPLOYED = False      # flip after `make smoke-agent` passes in Cloud Shell (Cell 7)

# THE DEPLOYED PEER, FROM HERE, WITH A CREDENTIAL PER REQUEST. Cloud Run IAM is the peer's front
# door, so every request - the card fetch AND each JSON-RPC post - carries an ID token minted for
# the peer's url. httpx calls auth_flow on every request, which is what keeps the hour-long fuse
# (7.3) from ever arming. RemoteA2aAgent uses this client for both hops (verified in the wheel).


class IdTokenAuth(httpx.Auth):
    """Mint AS a roster member, for one audience, on every request."""

    def __init__(self, audience: str, service_account: str = MEMBER_SA):
        self.audience, self.service_account = audience, service_account

    def auth_flow(self, request):
        request.headers["Authorization"] = f"Bearer {id_token_as(self.service_account, self.audience)}"
        yield request


if DEPLOYED:
    authed = httpx.AsyncClient(auth=IdTokenAuth(AGENT_URL), timeout=180)
    remote_peer = RemoteA2aAgent(name="documind_peer_cloud", description="DocuMind's document agent, on Cloud Run.",
                                 agent_card=f"{AGENT_URL}/.well-known/agent-card.json", httpx_client=authed)
    cloud_root = LlmAgent(name="orchestrator_cloud", model="gemini-3.6-flash",
                          instruction="Delegate every question about documents to documind_peer_cloud and relay its answer verbatim.",
                          sub_agents=[remote_peer])
    for line in await run_turn(cloud_root, "After how many years of continuous service does gratuity become payable?"):
        print(" ", line)
    # And the same hop without the ADK wrapper: the raw task, the way the smoke test sends it.
    task, answer = send(AGENT_URL, "What notice does the ACME MSA require to terminate for convenience?",
                        client=httpx.Client(auth=IdTokenAuth(AGENT_URL), timeout=180))     # lk-12: 90 days
    print("\nraw task:", task.get("status", {}).get("state"), "|", answer[:200])
else:
    print("DEPLOYED=False - deploy the peer from Cloud Shell (Cell 7), run make smoke-agent, then flip the switch")


## Cell 9: The roster refusal, relayed
The peer's account is on acme's roster only. Ask for zeta and the MCP server refuses; the peer can only say so.


In [ ]:
# THE ROSTER REFUSAL, RELAYED. The peer's account sits on acme's roster only (make roster, sa.tf).
# A task that names zeta makes the peer pass tenant='zeta' to the MCP server, the server checks the
# roster and refuses, and the peer says so - it cannot do otherwise, because the refusal is the tool's
# result. A2A carried no identity that could change that: the peer speaks to the lane as itself.
base, client = (AGENT_URL, httpx.Client(auth=IdTokenAuth(AGENT_URL), timeout=180)) if DEPLOYED else (LOCAL, None)
task, answer = send(base, "For tenant zeta: what is the per-trip cap on domestic travel reimbursement?", client=client)
print("state :", task.get("status", {}).get("state"))
print("answer:", answer[:300])
print("\nrefused by the roster:", any(w in answer.lower() for w in ("roster", "not on", "refus", "cannot", "not a member", "no access")))
# Run locally, the peer minted as documind-ui-sa - which IS on zeta's roster - so the local answer is
# Zeta's Rs 25,000. That difference is the whole identity story of this lesson in one line.


## Cell 10: A2A and MCP, side by side


In [ ]:
comparison = {
    "MCP (Module 7)": {"connects": "agent -> tools and data", "unit": "a tool call, stateless",
                       "identity": "the caller's token, verified by the server; the roster decides",
                       "in DocuMind": "documind-mcp: retrieve, list_documents, corpus_stats, calculate_processing_cost"},
    "A2A (this lesson)": {"connects": "agent <-> agent", "unit": "a task, with a lifecycle",
                          "identity": "IAM at the door; the peer speaks to the lane as ITSELF",
                          "in DocuMind": "documind-agent: a card, JSON-RPC at /, skills learned from the MCP server"},
}
for protocol, d in comparison.items():
    print(f"\n{protocol}")
    for k, v in d.items():
        print(f"  {k:10} {v}")

states = {"in progress": ["submitted", "working"], "waiting": ["input-required", "auth-required"],
          "terminal": ["completed", "failed", "canceled", "rejected"]}
print("\ntask states:", states)
print("\nUse BOTH: MCP for the vertical (an agent to its tools), A2A for the horizontal (an agent to its peers).")


## Stop the local peer


In [ ]:
peer_proc.terminate()
print("local peer stopped")


## Three protocols, three identities

```
you (Colab, mints as ui-sa) --A2A: ID token for documind-agent--> documind-agent (Cloud Run, runs as documind-agent-sa)
                                                                        |  McpToolset: ID token for documind-mcp, minted per call
                                                                        v
                                                                  documind-mcp  (verifies agent-sa, checks acme's roster)
                                                                        |  retrieve() as documind-mcp-sa
                                                                        v
                                                                  documind-api  (checks mcp-sa on acme's roster) -> answer + citations
```

A2A carries no identity of its own: IAM decides who may call the peer, and the peer speaks to the lane as itself. That is why a peer is scoped to one tenant, why it needs no tenant argument, and why a task naming another tenant is the server's refusal and not the peer's.

## ✅ Lesson 8.4 complete — and Module 8's agent surfaces with it
- ✅ The peer's code, requirements and Dockerfile written into the kit; no kit import, provable from the image
- ✅ The peer run here over the deployed MCP server; its card; one task asserted to have come from the corpus
- ✅ The peer as an ADK sub-agent, delegated to by a root
- ✅ The deploy block that is `commands/lesson-8.4.sh`; the deployed peer reached with a credential per request
- ✅ The zeta refusal relayed, and why the local and deployed peers answer it differently
